# Try TheSystem — Probe-256

Try automatic routing with four decision JSON layouts and four generation prompts.
Then make a decision and ask the same model to explain it.

The frozen Qwen backbone feeds a classifier with one 256-unit hidden layer. There are no LoRA modules in this model. The selected checkpoint is seed 42, epoch 6. Both release models use the same selected diverse-42 `<FAST>` gate.


## Before you start

Select **Runtime → Change runtime type → latest runtime version → NVIDIA GPU**.
Use Linux, Python 3.12 or 3.13 and native BF16 support. A **24 GB GPU may be tried**;
there is no minimum-memory cutoff. This capacity has not yet been validated for
these demos. Free T4 runtimes lack native BF16 support and are unsupported.

BF16 weights alone use about 18 GB. Loading and requests need extra GPU memory,
which depends on the prompt and response length. Setup reports total/free memory;
**READY** confirms the software and GPU checks, not that the model will fit.
Keep only one model loaded. Allow about 18 GB of disk space for the backbone plus caches.

Start with a fresh runtime. This demo reuses Colab's PyTorch 2.14.x and NumPy. Setup installs only the inference
packages it needs, without compiling CUDA extensions or running training checks.
The selected weights and prompt format are unchanged. Built-in PyTorch operations
can be slower and may give small numerical differences from the research kernels.
Use notebook 03 for reproducing the published measurements.

Run the cells in order. During private staging, enter a Hugging Face **read token**
from an account with access to this model repository. Keep the token private.


## 1. Prepare runtime

Run this cell and enter your **Hugging Face read token** in its hidden prompt
([token settings](https://huggingface.co/settings/tokens)). It checks the GPU,
installs missing inference packages, and verifies the downloaded release wheel.
Progress and elapsed-time updates appear every 20 seconds; details are saved in
`thesystem-prepare-runtime.log`.

Continue when it says **READY**. A fresh Colab session normally needs no restart.
If this session already imported an older package, setup will explain how to
restart once and rerun the cell. No compiler setup or separate NumPy upgrade is needed.


In [ ]:
# @title Prepare runtime
RUNTIME_READY = False
import getpass, hashlib, os, sys
from urllib.request import Request, urlopen
from urllib.error import HTTPError, URLError

PREPARE = {'repo_id': 'Lbuk/TheSystem-9B-Probe-256', 'revision': '39ce46e600a9ad7a2d5bf216b17bef71fe40b599', 'sha256': 'a0d1904552b5296ba7c3555fac79bb770b1a4efb75005a5d57e3580b7b3574bc'}
print("Prepare runtime: checking Python, then opening private setup access…", flush=True)
if sys.version_info[:2] not in ((3, 12), (3, 13)):
    raise RuntimeError("Use Python 3.12 or 3.13: Colab Runtime → Change runtime type → the latest runtime version.")
os.environ["HF_TOKEN"] = getpass.getpass("Hugging Face read token (hidden; Enter only for public access): ").strip()
print("Downloading the small, pinned setup helper (30-second timeout)…", flush=True)
url = f"https://huggingface.co/{PREPARE['repo_id']}/resolve/{PREPARE['revision']}/prepare_runtime.py"
headers = {"Authorization": f"Bearer {os.environ['HF_TOKEN']}"} if os.environ["HF_TOKEN"] else {}
try:
    with urlopen(Request(url, headers=headers), timeout=30) as response:
        helper = response.read()
except HTTPError as error:
    raise RuntimeError("Setup download failed. For HTTP 401/403, rerun this cell and paste a valid read token for this private repository. "
                       f"HTTP status: {error.code}. Your browser login is separate from Colab.") from None
except (URLError, TimeoutError) as error:
    raise RuntimeError("Setup helper download failed or timed out. Check the connection and rerun Prepare runtime.") from None
if hashlib.sha256(helper).hexdigest() != PREPARE["sha256"]:
    raise RuntimeError("Setup helper checksum mismatch. Stop and obtain a fresh release notebook.")
_setup = {"__name__": "thesystem_demo_setup"}
exec(compile(helper, "prepare_runtime.py", "exec"), _setup)
visible_progress = _setup["visible_progress"]
torch = _setup["prepare_runtime"]("probe")

RUNTIME_READY = True

## 2. Load Probe-256

Download only this model's release assets and the pinned upstream Qwen backbone.
The loader verifies the immutable release manifest and weights. Download progress
is shown; the first load is slower than later cached loads.

This demo uses the built-in inference operations. No training or gradient checks run.

If GPU memory runs out, the cell stops with recovery instructions. After a loading
failure, restart the session to clear partial weights. For a request, try a shorter
prompt or fewer `max_new_tokens`; if it still fails, use a larger GPU. The demo
never silently truncates input, quantizes weights or moves layers to the CPU.
Rerunning the load cell reuses the same already-loaded model.


In [ ]:
if not globals().get("RUNTIME_READY", False):
    raise RuntimeError("Run Prepare runtime until it reports READY before loading the model.")

from thesystem import load_model
from thesystem.artifacts import resolve_model_resources

if globals().get("model") is not None:
    if getattr(model, "variant", None) != "probe":
        raise RuntimeError("A different model is already loaded. Restart the session before loading this model.")
    print("Reusing the loaded model; continue to the examples.")
else:
    with visible_progress("Download and load the selected model"):
        RESOURCE_ROOT = resolve_model_resources("probe")
        model = load_model("probe", root=RESOURCE_ROOT, execution="inference")
    print("Loaded the selected Probe-256 checkpoint with the shared <FAST> gate.")


## 3. Decision mode

Try four JSON layouts. Each asks for exactly one supplied answer. The API receives JSON as prompt text
and a separate `choices` map matching the codes and labels in that text. Use 2–18 choices with consecutive
codes starting at A.

The training data for this release had no examples in the **Noul** JSON format (`type: "noul"`).
The next release will include Noul examples in training.

Every example uses `mode="auto"`. **Answer** shows the mode the gate selected and the result.
The `expected_mode` label appears in **Metadata** for comparison, alongside response time
and the `<FAST>` probability of choosing decision mode.
Edit a case or instruction and rerun its cell. To inspect extra fields, enter `result` in a new cell.


In [ ]:
import json
from contextlib import redirect_stdout
from io import StringIO
from time import perf_counter

def show_answer(result):
    print(f"Selected mode: {result['mode']}")
    if result["mode"] == "decision":
        print(f"Selected choice: {result['selected_code']} — {result['answer']}")
    elif result["mode"] == "generation":
        print(f"Response: {result['answer']}")
    else:
        print("The gate selected decision mode, which requires a choices mapping.")
    if result.get("hit_limit"):
        print("The generated response reached its token limit.")

def run_example(label, request, *, expected_mode, choices=None, metadata=None):
    prompt = request if isinstance(request, str) else json.dumps(request, ensure_ascii=False, indent=2)
    started = perf_counter()
    # Capture progress so the answer leads the cell output.
    with redirect_stdout(StringIO()), visible_progress(label):
        result = model.respond(prompt, choices=choices, mode="auto", max_new_tokens=128)
    elapsed = perf_counter() - started

    print("Answer" if metadata is None else f"\n{label}")
    show_answer(result)
    details = (
        f"{label}\n"
        f"<FAST> decision probability: {result['gate_probability']:.2%} (threshold: 50%)\n"
        f"Expected mode: {expected_mode} · Response time: {elapsed:.1f} s"
    )
    if metadata is None:
        print(f"\nMetadata\n{details}")
    else:
        metadata.append(details)
    return {key: value for key, value in result.items()
            if key not in {"trace", "routing_trace", "generation_trace"}}


In [ ]:
team_choices = {"A": "Billing", "B": "Technical support"}
team_request = {
    "context": "Billing handles invoices. Technical support handles API access. A customer's API key was rejected.",
    "request": "Which team should handle this ticket? Choose one of the listed teams.",
    "alternatives": team_choices,
}
result = run_example("Flat object with a choice map", team_request, expected_mode="decision", choices=team_choices)


In [ ]:
priority_choices = {"A": "Routine", "B": "Soon", "C": "Immediate"}
priority_request = {
    "context": "A typo is Routine, a slow page is Soon, and a full outage is Immediate. The site is down for everyone.",
    "question": "What is the incident priority?",
    "options": [{"code": code, "label": label} for code, label in priority_choices.items()],
    "instruction": "Select one option and return only its code.",
}
result = run_example("Array of labeled options", priority_request, expected_mode="decision", choices=priority_choices)


In [ ]:
refund_choices = {"A": "Eligible", "B": "Ineligible"}
refund_request = {
    "assessment_job": {
        "payload": {
            "evidence": "Refunds are allowed within 7 days. The customer purchased the item 4 days ago.",
            "query": "Is the customer eligible for a refund?",
        },
        "response_menu": refund_choices,
        "operation": "Make one selection and return only its answer code.",
    },
}
result = run_example("Nested assessment object", refund_request, expected_mode="decision", choices=refund_choices)


In [ ]:
desk_choices = {"A": "Finance", "B": "Engineering", "C": "General"}
desk_request = {
    "state": "Receipts go to Finance, bug reports to Engineering, and other documents to General. A bug report arrived.",
    "questions": {
        "destination": {
            "type": "choice",
            "instructions": "Which desk receives this document?",
            "criteria": list(desk_choices.values()),
        },
    },
    "requested_question": "destination",
    "candidate_codes": desk_choices,
    "answer_encoding": "Return only the code for the chosen desk.",
}
result = run_example("Named question with a separate code map", desk_request, expected_mode="decision", choices=desk_choices)


## 4. Generation mode

The first two prompts are plain text. The next two are JSON with writing instructions,
including one with an `options` list of topics to explain. All four ask the model to produce prose.


In [ ]:
writing_prompt = "In two sentences, reply to Maya about her rejected API key and ask her to check if it is active."
result = run_example("Customer reply from plain text", writing_prompt, expected_mode="generation")


In [ ]:
email_prompt = "Write a quick email to Sam saying I can attend the meeting tomorrow at 2 p.m."
result = run_example("Meeting confirmation from plain text", email_prompt, expected_mode="generation")


In [ ]:
reply_request = {
    "request": "Write a warm two-sentence reply to the customer.",
    "ticket": {"customer": "Maya", "issue": "The shipment is one day late."},
    "include": ["an apology", "the next update will arrive tomorrow"],
}
result = run_example("Customer reply from JSON", reply_request, expected_mode="generation")


In [ ]:
guide_request = {
    "request": "Write a short onboarding note explaining when to use each support channel.",
    "audience": "New teammates",
    "options": [
        {"channel": "Chat", "use": "Quick questions during business hours"},
        {"channel": "Email", "use": "Requests that need attachments or a written record"},
    ],
}
result = run_example("Guide from JSON with an options list", guide_request, expected_mode="generation")


## 5. Decide, then explain

Each iteration revisits a case from section 3, then sends the chosen code and answer as a JSON record
in a second request to the same model. That call asks Qwen to explain the supplied choice using the case
facts. The generated explanation may disagree with the selection.


In [ ]:
cases = [("Support team", team_request, team_choices), ("Incident priority", priority_request, priority_choices)]
metadata = []
print("Answer")
for label, request, choices in cases:
    decision = run_example(label, request, expected_mode="decision", choices=choices, metadata=metadata)
    if decision["mode"] != "decision":
        print("Skipping the explanation because this request returned no selected choice.")
        continue

    decision_record = {
        "context": request["context"],
        "selected_code": decision["selected_code"],
        "selected_answer": decision["answer"],
    }
    explanation_request = {
        "request": (
            "Write two sentences explaining the recorded decision using the case facts. "
            "If the facts conflict with the selected answer, describe the conflict."
        ),
        "decision": decision_record,
    }
    run_example(f"{label} explanation", explanation_request, expected_mode="generation", metadata=metadata)
print("\nMetadata\n" + "\n\n".join(metadata))


## 6. Finish when you are ready

**Run all leaves the model available for experimentation.** When finished, set
`RELEASE_RUNTIME = True` below and run that cell. It frees the model and releases
a Colab runtime; it does not shut down an ordinary local kernel.


In [ ]:
RELEASE_RUNTIME = False  # Set True and run this cell when you have finished experimenting.
if RELEASE_RUNTIME:
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    try:
        from google.colab import runtime
    except ImportError:
        print("Model unloaded. This local kernel stays open.")
    else:
        runtime.unassign()
else:
    print("The model stays loaded. Edit an example and rerun its cell to keep exploring.")
